# 促銷分析｜優惠券與折扣效益評估

商業問題：  
優惠券使用與客單價影響分析  
分析方法：  
- 區分使用與未使用優惠券的訂單
- 比較兩組平均訂單金額
- 使用獨立樣本 t 檢定，判斷客單價差異是否達統計顯著

In [4]:
import pandas as pd
from scipy import stats
sales=pd.read_csv("sales.csv")
group_coupon=sales[sales["Coupon_Code"].notna()]["Order_Value"]
group_no_coupon=sales[sales["Coupon_Code"].isna()]["Order_Value"]
t_stat,p_val=stats.ttest_ind(group_coupon,group_no_coupon,equal_var=False)
print(f"有優惠券平均金額:{group_coupon.mean():.2f}")
print(f"無優惠券平均金額:{group_no_coupon.mean():.2f}")
print(f"t 統計量 (t-statistic):{t_stat:.4f}")
print(f"p 值 (p-value):{p_val:.4f}")
if p_val<0.05:
    print("結論: p<0.05，兩組客單價存在統計上的顯著差異。")
else:
    print("結論:p>=0.05，兩組客單價無顯著差異。")


有優惠券平均金額:24299.94
無優惠券平均金額:23884.46
t 統計量 (t-statistic):2.1948
p 值 (p-value):0.0282
結論: p<0.05，兩組客單價存在統計上的顯著差異。


分析結果：  
t 檢定結果 p 值為 0.0282 ，小於 0.05，表示使用優惠券與未使用優惠券的訂單，其平均訂單金額差異達統計顯著。使用優惠券的訂單平均金額為 24,299.94 ，未使用優惠券為 23,884.46 ，差距約 415 元。雖然兩組平均訂單金額的差異達統計顯著，但實際金額差距幅度有限，因此優惠券使用與訂單金額之間雖存在統計上的差異，實務上的差異程度仍不大。

商業問題：  
各優惠券折扣效益與成本分析  
分析方法：  
- 計算各優惠券的營收與折扣成本  
- 以`（優惠券訂單營收－優惠券成本） / 優惠券成本`計算 ROI
- 比較不同優惠券的成本效益

In [ ]:
import pandas as pd
sales=pd.read_csv("sales.csv")
coupon_sales=sales.dropna(subset=['Coupon_Code'])
coupon_summary=coupon_sales.groupby("Coupon_Code").agg(
    Total_Revenue=("Order_Value","sum"),
    Total_Cost=("Coupon_Discount","sum")
).reset_index()
coupon_summary["ROI"]=(coupon_summary["Total_Revenue"]-coupon_summary["Total_Cost"])/coupon_summary["Total_Cost"]
result=coupon_summary.sort_values(by="ROI", ascending=False)
print(result)

  Coupon_Code  Total_Revenue   Total_Cost         ROI
1      FLAT50   3.044658e+08    628950.00  483.085795
0   DIWALI100   3.089226e+08   1260900.00  244.001697
2      SAVE10   6.061043e+08  60610436.53    8.999999


分析結果：  
以 ROI 衡量，優惠券 FLAT50 的效益最高（約 483 倍）， DIWALI100 次之（約 244 倍）， SAVE10 雖帶動的訂單營收最高，但優惠券成本也較高， ROI 僅約 9 倍，效益明顯偏低。以本次分析的成本效益指標來看， FLAT50 在優惠券成本與訂單營收的相對表現較佳； SAVE10 雖具有較高的營收規模，但相對投入的折扣成本也較高。

商業問題：  
促銷期間新舊客比例分析  
分析方法：  
- 判斷各客戶是否為首次下單
- 篩選使用優惠券的訂單
- 計算促銷訂單中新客占比

In [5]:
import pandas as pd
sales=pd.read_csv("sales.csv")
sales["Order_Date"]=pd.to_datetime(sales["Order_Date"],format="%d/%m/%Y")
sales=sales[sales["Order_Status"]=="Delivered"].copy()
sales=sales.sort_values(["Customer_ID","Order_Date"])
sales["Is_First_Order"]=sales.groupby("Customer_ID").cumcount()==0
promo_orders=sales[sales["Coupon_Code"].notna()]
promo_order_count=len(promo_orders)
new_order_count=promo_orders["Is_First_Order"].sum()
new_cust_ratio=promo_orders["Is_First_Order"].mean()
print(f"促銷訂單筆數：{promo_order_count:,}")
print(f"其中首購訂單筆數：{new_order_count:,}")
print(f"促銷訂單新客占比：{new_cust_ratio:.2%}")


促銷訂單筆數：40,156
其中首購訂單筆數：7,897
促銷訂單新客占比：19.67%


分析結果：  
以客戶首筆有效訂單判定新客，並篩選使用優惠券的促銷訂單，共 40,156 筆，其中首購訂單 7,897 筆，新客占比為 19.67%。結果顯示，促銷訂單約 80% 來自既有客戶，促銷活動的訂單主要由回購客戶構成。